# Notebook 4: Photon Statistics — Zero-Delay Correlations of Quantum Light

## What are we exploring?

The second-order coherence function $g^{(2)}(0)$ is a **central zero-delay
photon-correlation observable** in quantum optics. It classifies photon
statistics into three regimes:

| Regime | g²(0) | Statistics | Example |
|--------|-------|-----------| --------|
| Anti-bunching | < 1 | Sub-Poissonian | Fock states |
| Random | = 1 | Poissonian | Coherent states (lasers) |
| Bunching | > 1 | Super-Poissonian | Thermal light, squeezed vacuum |

In this notebook, we bring together all four fundamental quantum states and
compare their photon statistics in one unified analysis.

## Conventions used in this notebook

- Natural units are used for the oscillator algebra: $\hbar=1$.
- `N` is the Hilbert-space dimension, so photon numbers run from `0` to `N-1`.
- `g^{(2)}(0)` is a zero-delay correlation diagnostic, not a complete state classifier.
- Analytic tails and observable convergence are used for truncation checks; never trust `sum(P)` alone.

## Setup: Create All Four States with $\langle n \rangle \approx 3$

In [1]:
from pathlib import Path
import sys

import numpy as np
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import qutip

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
FIG_DIR = PROJECT_ROOT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
from qo_utils import (
    photon_distribution,
    plot_photon_distribution,
    plot_wigner,
    wigner_normalization,
    mean_photon_number,
    photon_variance,
    compute_g2_zero,
    mandel_Q,
    coherent_tail,
    thermal_tail,
    squeezed_vacuum_tail,
    cutoff_from_tail,
    squeezed_wigner_extent,
)

plt.rcParams.update({
    'font.size': 12, 'axes.labelsize': 14, 'axes.titlesize': 16,
    'figure.figsize': (8, 5), 'figure.dpi': 150, 'savefig.dpi': 300,
    'text.usetex': False, 'mathtext.fontset': 'stix', 'font.family': 'STIXGeneral',
})

N = 80  # Required for the matched squeezed state with <n> = 3
a = qutip.destroy(N)

# Target: all states with <n> ~ 3 for fair comparison
n_target = 3

# 1. Fock state |3>
fock_state = qutip.basis(N, n_target)

# 2. Coherent state |alpha=sqrt(3)> -> <n> = 3
coherent_state = qutip.coherent(N, np.sqrt(n_target))

# 3. Squeezed vacuum with <n> = sinh^2(r) ~ 3 -> r = arcsinh(sqrt(3))
r_squeeze = np.arcsinh(np.sqrt(n_target))
assert squeezed_vacuum_tail(N, r_squeeze) < 1e-5, "Increase N for the matched squeezed state"
squeezed_state = qutip.squeeze(N, r_squeeze) * qutip.basis(N, 0)

# 4. Thermal state with n_bar = 3
thermal_state = qutip.thermal_dm(N, n_target)

states = {
    'Fock |3>': fock_state,
    'Coherent': coherent_state,
    'Squeezed': squeezed_state,
    'Thermal': thermal_state,
}

# Verify all have <n> ~ 3
print("=== State Verification ===\n")
for name, state in states.items():
    mn = mean_photon_number(state, a)
    print(f"{name:<15}: <n> = {mn:.3f}")

=== State Verification ===

Fock |3>       : <n> = 3.000
Coherent       : <n> = 3.000
Squeezed       : <n> = 3.000
Thermal        : <n> = 3.000


## THE Master Comparison Table

This table is the central deliverable of the single-mode analysis. All four
states have matched $\langle n \rangle \approx 3$, yet they have completely
different photon statistics.